# Introduction

## Remarks regarding copyright & special thanks

1. The knowledge portrayed is based on the remarkable book "Large Language Models selbst programmieren" (copyright is with Rheinwerk Verlag, German translation of "Build a Large Language Model (From Scratch)", copyright is with Maning Publications), written by Sebastian Raschka. 
2. Special thanks & shout out: I am very grateful for such a book that despite all the AI fluff and AI hype conveys what LLM do under the hood and how they can be build.
3. I declare: My interest is solely in learning. I am not using ideas and concepts laid out in this book for proprietary economical interests.

## Sources

- Raschka, Sebastian (2025). Large Language Models selbs programmieren: Mit Python und PyTorch ein eigenes LLM entwickeln. dpunkt.verlag (Rheinwerk Verlag): Bonn.
- Text the LLMs is trained on: https://en.wikisource.org/wiki/The_Verdict

# Imports

In [1]:
import torch
import re
import pandas as pd
import tiktoken
from torch.utils.data import Dataset, DataLoader

# Hardware used for compute

In [2]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Current available device: {device}")

Current available device: mps


# (1) Working with Text Data

## Import text

In [3]:
with open("the-verdict.txt", "r", encoding="utf-8") as file:
    raw_text = file.read()

In [4]:
print("Total number of chars:", len(raw_text))
print(raw_text[:100])

Total number of chars: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


## Tokenizing text

### testing tokenizer

In [5]:
# input text
text = "Hello, world. Is this-a test?"

# tokenizing schema
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)

# tokenized text
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this-a', 'test', '?']


### Preprocessing (applying tokenizer on imported text)

In [6]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item for item in preprocessed if item.strip()]
print(f"Token length of imported text: {len(preprocessed)}")
n = 30
print(f"\nFirst {n} tokens: {preprocessed[:n]}")

Token length of imported text: 4690

First 30 tokens: ['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


## Converting tokens into Token-IDs

In [7]:
all_tokens = sorted(set(preprocessed))
vocab_size = len(all_tokens)
print(f"Size of words: {vocab_size}")
print(f"\nFirst {n} alphabetically ordered tokens {all_tokens[:n]}")

Size of words: 1130

First 30 alphabetically ordered tokens ['!', '"', "'", '(', ')', ',', '--', '.', ':', ';', '?', 'A', 'Ah', 'Among', 'And', 'Are', 'Arrt', 'As', 'At', 'Be', 'Begin', 'Burlington', 'But', 'By', 'Carlo', 'Chicago', 'Claude', 'Come', 'Croft', 'Destroyed']


In [8]:
vocab = {token: ID for ID, token in enumerate(all_tokens)}

In [9]:
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token: ID for ID, token in enumerate(all_tokens)}

In [10]:
df = pd.DataFrame(vocab.items(), columns=["Token", "Token ID"])
df

,Token,Token ID
0,!,0
1,"""",1
2,',2
3,(,3
4,),4
...,...,...
1127,younger,1127
1128,your,1128
1129,yourself,1129
1130,<|endoftext|>,1130


## How to build a tokenizer (V2)

In [11]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        # saves vocabulary as class attribute to access during Encode-Decode-methods
        self.str_to_int = vocab
        # creates inverse vocabulary (Token IDs to original text token)
        self.int_to_str = {integer:string for string,integer in vocab.items()}

    # transforms input text to Token IDs
    def encode(self, text):
        preprocessed = re.split(r'([,.;:?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]

        # unknown words are replaced by <|unk|> token
        preprocessed = [item if item in self.str_to_int else "<|unk|>" for item in preprocessed]
        
        ID_list = [self.str_to_int[string] for string in preprocessed]
        return ID_list

    # converts Token IDs back to text
    def decode(self, ID_list):
        text = " ".join([self.int_to_str[integer] for integer in ID_list])
        # removes space before defined punctuation marks
        text = re.sub(r'\s+([,.;:?!"()\'])', r'\1', text)
        return text

### Testing tokenizer (V2)


In [12]:
tokenizer = SimpleTokenizerV2(vocab)

text = """
"It's the last he painted, you know," Mrs. Gisburn said with pardonable pride. 
"The last but one," she corrected herself--"but the other doesn't count, because he destroyed it."
"""

ID_list = tokenizer.encode(text)
print(f"Text to Token IDs:\n{ID_list}")
print(f"\n\nToken IDs to text:\n{tokenizer.decode(ID_list)}")

Text to Token IDs:
[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7, 1, 93, 602, 239, 729, 5, 1, 876, 291, 542, 6, 1, 239, 988, 735, 356, 2, 970, 294, 5, 205, 533, 330, 585, 7, 1]


Token IDs to text:
" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride." The last but one," she corrected herself --" but the other doesn' t count, because he destroyed it."


### Testing <|endoftext|> token

In [13]:
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
print(text)

tokenizer = SimpleTokenizerV2(vocab)
ID_list = tokenizer.encode(text)
print(f"Text to Token IDs:\n{ID_list}")

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.
Text to Token IDs:
[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]


In [14]:
from importlib.metadata import version
print(f"tiktoken version: {version('tiktoken')}")

tiktoken version: 0.12.0


### Using `tiktoken`

In [15]:
tokenizer = tiktoken.get_encoding("gpt2")
text = "Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace."

# encoding
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

# decoding
strings = tokenizer.decode(integers)
print(strings)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]
Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.


In [16]:
tokens = [tokenizer.decode([i]) for i in integers]

df = pd.DataFrame({"Token": tokens, "Token ID": integers})
df

,Token,Token ID
0,Hello,15496
1,",",11
2,do,466
3,you,345
4,like,588
5,tea,8887
6,?,30
7,,220
8,<|endoftext|>,50256
9,In,554


## Data sampling

In [17]:
with open("the-verdict.txt", "r",encoding="utf-8") as file:
    raw_text = file.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [18]:
enc_sample = enc_text[50:]

# context size determines how many tokens as part of input are taken into account
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(f"x:{x}")
print(f"y:     {y}")

x:[290, 4920, 2241, 287]
y:     [4920, 2241, 287, 257]


In [19]:
# prediction of next word
for i in range(1,context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "---->", desired)

print("\n")    

# conversion of Token IDs into text
for i in range(1,context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


In [20]:
enc_sample[:10]

[290, 4920, 2241, 287, 257, 4489, 64, 319, 262, 34686]

### Dataset for stacked inputs and targets

In [21]:
class GPTDatasetV1(Dataset):
    def __init__(self, text, tokenizer, max_length, stride):
        self.input_IDs = []
        self.target_IDs = []

        # tokenizing of whole text
        token_IDs = tokenizer.encode(text)

        # using a string window to cut text into overlapping sequences of `max_length`
        for i in range(0, len(token_IDs) - max_length, stride):
            input_chunk = token_IDs[i:i + max_length]
            target_chunk = token_IDs[i + 1:i + max_length + 1]
            self.input_IDs.append(torch.tensor(input_chunk))
            self.target_IDs.append(torch.tensor(target_chunk))

    # total of lines in dataset
    def __len__(self):
        return len(self.input_IDs)

    # single line in dataset
    def __getitem__(self, index):
        return self.input_IDs[index], self.target_IDs[index]

In [22]:
def create_dataloader_v1(
    text, batch_size=4, max_length=256, stride=128,
    shuffle=True, drop_last=True, num_workers=0
):
    # init of tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")
    # creates dataset
    dataset = GPTDatasetV1(text, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last, # last batch dropped, if short than defined `batch_size` (reduces loss peaks during training)
        num_workers=num_workers # num of CPU-processes used during preprocessing
    )
    return dataloader
    
    

In [23]:
with open("the-verdict.txt", "r", encoding="utf-8") as file:
    raw_text = file.read()

dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
# converts dataloader into Python iterator to call next entry with `next()`
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [24]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


### Sample of `batch_size` > 1

#### `max_length=4` and `stride=1`: high token overlap

In [25]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print(f"Inputs:\n\t{inputs}")
print(f"Targets:\n\t{targets}")

Inputs:
	tensor([[   40,   367,  2885,  1464],
        [  367,  2885,  1464,  1807],
        [ 2885,  1464,  1807,  3619],
        [ 1464,  1807,  3619,   402],
        [ 1807,  3619,   402,   271],
        [ 3619,   402,   271, 10899],
        [  402,   271, 10899,  2138],
        [  271, 10899,  2138,   257]])
Targets:
	tensor([[  367,  2885,  1464,  1807],
        [ 2885,  1464,  1807,  3619],
        [ 1464,  1807,  3619,   402],
        [ 1807,  3619,   402,   271],
        [ 3619,   402,   271, 10899],
        [  402,   271, 10899,  2138],
        [  271, 10899,  2138,   257],
        [10899,  2138,   257,  7026]])


#### `max_length=4` and `stride=4`: no token overlap

with value of `stride` corresponding to value of `max_length` overlaps between batches are avoided (overlaps increase overfitting)

In [26]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print(f"Inputs:\n\t{inputs}")
print(f"Targets:\n\t{targets}")

Inputs:
	tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Targets:
	tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


## Creating Token-Embeddings

embedding weights are randomly initialized (starting point for LLM learning process)

### Simple example of randomly initialized Embedding Weights of dimension 3

In [27]:
input_IDs = torch.tensor([2,3,5,1])

vocab_size = 6
output_dim = 3

torch.manual_seed(42)
embedding_layer = torch.nn.Embedding(vocab_size,output_dim)
# weight matrix of embedded layer (size of 6, due to `vocab_size`, with a dimension of 3 for each)
print(embedding_layer.weight)

# weight matrix applied on one Token ID, i.e. ID 3
print(embedding_layer(torch.tensor([3])))

# weight matrix applied on input IDs 
print(embedding_layer(input_IDs))

Parameter containing:
tensor([[ 1.9269,  1.4873, -0.4974],
        [ 0.4396, -0.7581,  1.0783],
        [ 0.8008,  1.6806,  0.3559],
        [-0.6866,  0.6105,  1.3347],
        [-0.2316,  0.0418, -0.2516],
        [ 0.8599, -0.3097, -0.3957]], requires_grad=True)
tensor([[-0.6866,  0.6105,  1.3347]], grad_fn=<EmbeddingBackward0>)
tensor([[ 0.8008,  1.6806,  0.3559],
        [-0.6866,  0.6105,  1.3347],
        [ 0.8599, -0.3097, -0.3957],
        [ 0.4396, -0.7581,  1.0783]], grad_fn=<EmbeddingBackward0>)


### 256-dim example

In [28]:
vocab_size = 50257
output_dim = 256
batch_size = 8
max_length = 4
shuffle = False

token_embedding_layer = torch.nn.Embedding(vocab_size,output_dim)
dataloader = create_dataloader_v1(raw_text, batch_size=batch_size, max_length=max_length, shuffle=shuffle)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

print(f"Token IDs:\n\t{inputs}")
print(f"\nInputs shape:\n\t{inputs.shape}")

token_embeddings = token_embedding_layer(inputs)
print(f"\nToken embeddings shape:\n\t{token_embeddings.shape}")

Token IDs:
	tensor([[   40,   367,  2885,  1464],
        [  286,   616,  4286,   705],
        [10197,   832,   262, 46475],
        [ 4150,     8,  3688,   284],
        [  271, 10899,   550,   366],
        [ 1021,   757,   438, 10919],
        [  314,  4752,   340,  6777],
        [  423,  4750,   326,  9074]])

Inputs shape:
	torch.Size([8, 4])

Token embeddings shape:
	torch.Size([8, 4, 256])


### Absolute embedding-approach of GPT models


In [29]:
# `context_length` equals input size supported by LLM 
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
# placeholder vector (sequence of nums up to maximum imput length (-1))
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)

# note: in practice, input text can be larger than supported context length

torch.Size([4, 256])


In [30]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)
print(f"Token embeddings:\n\t{token_embeddings}")
print(f"\nPositional embeddings:\n\t{pos_embeddings}")
print(f"\nInput embeddings (`token_embeddings + pos_embeddings`):\n\t{input_embeddings}")

torch.Size([8, 4, 256])
Token embeddings:
	tensor([[[-0.2263,  1.5471,  1.7993,  ..., -1.0731, -0.6847, -1.1511],
         [ 0.0424,  0.6706,  3.0505,  ...,  1.7153,  0.1248,  0.5104],
         [-0.0822,  0.2824, -0.0252,  ..., -1.3636,  0.8076, -0.5555],
         [-0.3119, -0.5069,  0.6564,  ...,  0.4261, -0.6862,  0.1792]],

        [[-0.4823, -0.2676, -0.4044,  ..., -0.7528, -1.3334, -0.8553],
         [ 1.9877,  0.0464,  1.3958,  ...,  0.2153,  0.3791,  1.4940],
         [-1.4902, -0.3452,  1.9843,  ..., -1.3788,  1.4019, -1.2577],
         [ 0.2288, -1.3381,  0.3087,  ...,  1.1380, -1.7751,  0.2028]],

        [[-1.7628,  0.0183,  0.6749,  ...,  0.7473,  0.7538,  2.2061],
         [ 0.2041, -0.7187,  0.2453,  ..., -0.4150, -0.4495,  1.3609],
         [-0.0527,  1.1761, -0.8884,  ...,  0.1635,  1.1442, -0.5673],
         [-0.0488,  1.1622,  0.8161,  ..., -1.0131, -0.8273,  1.5470]],

        ...,

        [[-0.8878, -0.4020, -0.7643,  ...,  2.5732,  1.3734, -1.8008],
         [ 0.0

# (2) Attention Mechanism

## Self-Attention Mechanism

### Dimensions used in this notebook

$$
n = 6, \quad d_{\text{model}} = 3, \quad d_k = 2
$$

### Tokenization

A text sequence is first converted into a sequence of token IDs:

$$
\text{text} \rightarrow [t_1, t_2, \dots, t_n]
$$

where \(t_i\) represents the integer ID of token \(i\).

### Token Embeddings

Each token ID is mapped to a vector using an embedding matrix:

$$
E \in \mathbb{R}^{V \times d_{\text{model}}}
$$

where  

- \(V\) = vocabulary size  
- \(d_{\text{model}}\) = embedding dimension.

The embedding sequence becomes

$$
X =
\begin{bmatrix}
x^{(1)} \\
x^{(2)} \\
\vdots \\
x^{(n)}
\end{bmatrix}
\in \mathbb{R}^{n \times d_{\text{model}}}
$$

---

### Positional Embeddings

Since transformers process tokens in parallel, positional information is added:

$$
x_i = e_i + p_i
$$

where

- \(e_i\) = token embedding  
- \(p_i\) = positional embedding.

The resulting matrix is

$$
X \in \mathbb{R}^{n \times d_{\text{model}}}
$$

---

### Query, Key, Value Projections

Three learned projection matrices transform the input embeddings:

$$
W_Q, W_K, W_V \in \mathbb{R}^{d_{\text{model}} \times d_k}
$$

The projections are

$$
Q = XW_Q
$$

$$
K = XW_K
$$

$$
V = XW_V
$$

with

$$
Q,K,V \in \mathbb{R}^{n \times d_k}
$$

### Attention Scores

Attention scores measure similarity between queries and keys:

$$
S = QK^T
$$

$$
S \in \mathbb{R}^{n \times n}
$$

Each entry

$$
s_{ij} = q_i \cdot k_j
$$

represents how much token \(i\) attends to token \(j\).

---

### Scaled Softmax

To stabilize training, scores are scaled:

$$
A =
\text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)
$$

where

$$
A \in \mathbb{R}^{n \times n}
$$

and each row sums to 1.

### Context Vectors

The attention weights are applied to the value matrix:

$$
Z = AV
$$

with

$$
Z \in \mathbb{R}^{n \times d_k}
$$

Each row

$$
z_i
$$

represents the contextualized representation of token \(i\).


### Final Attention Formula

The complete self-attention operation is therefore

$$
\text{Attention}(Q,K,V) =
\text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

## Simple Self-Attention-Mechanism w/o trainable weights

In [31]:
# input-sequence
inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89],
        [0.55, 0.87, 0.66],
        [0.57, 0.85, 0.64],
        [0.22, 0.58, 0.33],
        [0.77, 0.25, 0.10],
        [0.05, 0.80, 0.55],
    ]
)

# second input token serves as query
query = inputs[1]
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)

print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


### Normalizing attention scores (based on input vector 2)

1. first a simple handcoded solution for normalizing attention weights
2. using a better alternative, namely Softmax function
   - better handling of extreme values,
   - better gradient attributes during training,
   - ensures that attention weights are always positive
     - (larger weights relate to greater importance)
     - instable regarding huge and small numbers (e.g., overflow)
3. safest option: using built-in Softmax function
   - handles huge and smalls values best

In [32]:
print("--------------------------------------------------")
print("Using self-coded normalization of attention weights:\n")

attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print(f"Attention weights:\n\t{attn_weights_2_tmp}")
print(f"Sum:\n\t{attn_weights_2_tmp.sum()}")

print("\n--------------------------------------------------")
print("Using self-coded Softmax:\n")

def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)
print(f"\nAttention weights:\n\t{attn_weights_2_naive}")
print(f"Sum:\n\t{attn_weights_2_naive.sum()}")

print("\n--------------------------------------------------")
print("Using built-in Softmax:\n")

attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print(f"\nAttention weights:\n\t{attn_weights_2}")
print(f"Sum:\n\t{attn_weights_2.sum()}")

--------------------------------------------------
Using self-coded normalization of attention weights:

Attention weights:
	tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum:
	1.0000001192092896

--------------------------------------------------
Using self-coded Softmax:


Attention weights:
	tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum:
	1.0

--------------------------------------------------
Using built-in Softmax:


Attention weights:
	tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum:
	1.0


### Computing context vector $ z^{(2)} $


$ z^{(2)}$ is the weighted sum of all input vectors$

In [33]:
# second input token is the query
query = inputs[1]
context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i] * x_i

print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


### How to compute all vecors
1. Compute Attention Scores
2. Compute Attention Weights
3. Compute Context Vectors

#### 1) Compute Attention Scores

In [34]:
attn_scores = torch.empty(6,6)

print("--------------------------------------------------")
print("Pretty slow!:")
for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)

print(f"Computing attentions scores based on all inputs:\n\t{attn_scores}")

print("\n--------------------------------------------------")
print("Much faster (same result)!:")

attn_scores = inputs @ inputs.T

print(f"Computing attentions scores based on all inputs:\n\t{attn_scores}")

--------------------------------------------------
Pretty slow!:
Computing attentions scores based on all inputs:
	tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

--------------------------------------------------
Much faster (same result)!:
Computing attentions scores based on all inputs:
	tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


#### 2) Compute Attention Weights

Attention weights define the extent to which a context vector depends on the different parts of its input (cf. Rascka (2025), p. 93).

Next step: Normalization of attention scores.

In [35]:
# normalization
attn_weights = torch.softmax(attn_scores, dim=-1)

print(f"Attention weights:\n\t{attn_weights}")

print(f"\nAll vector / row sum:\n\t{attn_weights.sum(dim=-1)}")

Attention weights:
	tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

All vector / row sum:
	tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


#### 3) Compute Context Vectors

In [36]:
all_context_vecs = attn_weights @ inputs
print(f"All context vectors for input vectors:\n\t{all_context_vecs}")

All context vectors for input vectors:
	tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


## Self-Attention-Mechanism w trainable weights ($W_q$, $W_k$, $W_v$)
- $W_q$ = *query* weight matrice
- $W_k$ = *key* weight matrice
- $W_v$ = *value* weight matrice

Weight params are learned coefficents that define the connections within the NN (cf. Rascka (2025), p. 93).

In [61]:
# example with second input / X^(2)

# second input
x_2 = inputs[1]
# size of input-embedding
d_in = inputs.shape[1]
# size of output-embedding
d_out = 2

# initializing weight matrices (W_query, W_key, W_value)
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

# computing query-vector, key-vector, value-vector
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value

print(f"Query vector:\t{query_2}\nKey vector:\t{key_2}\nValue vector:\t{value_2}\n\n(based on d_out={d_out})")

print("--------------------------------------------------")

# even if just one context vector (z^(2)) is to be computed,
# key and value vector for all imput elements are needed (cf. Rascka (2025), p. 93)
keys = inputs @ W_key
values = inputs @ W_value

print(f"keys.shape:\t{keys.shape}")
print(f"values.shape:\t{values.shape}")

Query vector:	tensor([0.4306, 1.4551])
Key vector:	tensor([0.4433, 1.1419])
Value vector:	tensor([0.3951, 1.0037])

(based on d_out=2)
--------------------------------------------------
keys.shape:	torch.Size([6, 2])
values.shape:	torch.Size([6, 2])


In [62]:
keys_2 = keys[1]
attn_score_22 = query_2.dot(keys_2)
print(f"Not normalized attention score:\n\t{attn_score_22}")

# all attention score for one query
attn_scores_2 = query_2 @ keys.T
print(f"Attention scores:\n\t{attn_scores_2}")

Not normalized attention score:
	1.8523845672607422
Attention scores:
	tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


### Attention weights

In [63]:
d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)

print(attn_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


### Context vector $z^{(2)}$

In [80]:
context_vec_2 = attn_weights_2 @ values
print(f"Context vector for z^(2):\ninputs (x^(2)):\n\t{inputs[1]}\n\ncontext vector (z^(2)):\n\t{context_vec_2}")

Context vector for z^(2):
inputs (x^(2)):
	tensor([0.5500, 0.8700, 0.6600])

context vector (z^(2)):
	tensor([0.3061, 0.8210])


### Self-Attention-Class

$$
\text{Attention}(Q,K,V) =
\text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

#### SelfAttention_v1 class

In [105]:
class SelfAttention_v1(torch.nn.Module):
    def __init__(self, d_in, d_out):
        # initializing trainable weight matrices
        super().__init__()
        self.W_query = torch.nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = torch.nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = torch.nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        queries = x @ self.W_query 
        keys = x @ self.W_key
        values = x @ self.W_value
        # computing attention scores > normalization with Softmax (attention weights)
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        # computing one context vector
        context_vec = attn_weights @ values
        return context_vec

##### Using `SelfAttention_v1` to compute context vectors $z^{(1)}$ to $z^{(n)}$

In [106]:
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(f"Context vector for inputs:\ninputs (x^(1) to x^(n)):\n{inputs}:\n\ncontext(z^(1) to z^(n)):\n{sa_v1(inputs)}")

Context vector for inputs:
inputs (x^(1) to x^(n)):
tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]]):

context(z^(1) to z^(n)):
tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


#### SelfAttention_v2

`torch.nn.Parameter(torch.rand(d_in, d_out))` replaced by `torch.nn.Linear(d_in, d_out, bias=qkv_bias)` for better weight-relation initialization schema, leads to a more stable and effective model training 

In [107]:
class SelfAttention_v2(torch.nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        # initializing trainable weight matrices
        super().__init__()
        self.W_query = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = torch.nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)
        # computing attention scores > normalization with Softmax (attention weights)
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        # computing one context vector
        context_vec = attn_weights @ values
        return context_vec

##### Using `SelfAttention_v2` to compute context vectors $z^{(1)}$ to $z^{(n)}$

In [108]:
torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(f"Context vector:\n{sa_v2(inputs)}")

Context vector:
tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


## Masking future words with causal attention

Attention weight add up to 1 for each line.

### Computing Attention Weights Softmax-function

In [109]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print(f"Total of attention weights:\n{attn_weights}")

Total of attention weights:
tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


### Mask

#### Option 1 Masking

Built-in PyTorch function `tril` zeroes every entry upwards from the diagonal

In [114]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(f"Simple mask:\n{mask_simple}")

Simple mask:
tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


##### Multiplying mask with attention weights

In [111]:
masked_simple = attn_weights * mask_simple

###### Masked Attention Weight

In [112]:
print(f"Masked Attention Weights:\n{masked_simple}")

Masked Attention Weights:
tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)


##### Normalization of masked Attention Weights

In [113]:
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(f"Normalized masked Attention Weights:\n{masked_simple_norm}")

Normalized masked Attention Weights:
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


#### Option 2 Masking

More efficient: using `triu` function and applying `softmax` to normalize!

In [117]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(f"Masked Attention Weights w inf values:\n{masked}")

attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=1)
print(f"\nNormalized masked Attention Weights:\n{attn_weights}")

Masked Attention Weights w inf values:
tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)

Normalized masked Attention Weights:
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


### Reduction of Overfitting by dropping out elements randomly

Typically used either after computation of Attention Weights or after applying them in value vectors.

#### Example with random 6x6 tensor

To balance the number of active elements all values (ones) remaining within thr matrix are up-scaled by the dropout rate: `1 / dropout = x`: `1 / 0.5 = 2`

In [125]:
torch.manual_seed(123)

# dropout ration of 50%
dropout = torch.nn.Dropout(0.5)
# creating a matrix with ones
ones = torch.ones(6,6)
print(f"Example of ones:\n{ones}")
print(f"\nDropout:\n{dropout(ones)}")

Example of ones:
tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])

Dropout:
tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


#### Masked Attention Weights with `dropout`

In [126]:
torch.manual_seed(123)
print(dropout(attn_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7599, 0.6194, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4921, 0.4925, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3966, 0.0000, 0.3775, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.3331, 0.3084, 0.3331, 0.0000]],
       grad_fn=<MulBackward0>)


## Causal Attention Class

In [133]:
class CausalAttention(torch.nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        # compared to previous SelfAttention class the "dropout layer" is new
        self.dropout = torch.nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )
    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1,2)
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )

        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec
        

In [138]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)

# resulting context vector is a 3-dim tensor;
# every token is presented by a 2-dim embedding
torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)
print(f"\ncontext_vec.shape:\n{context_vecs.shape}")

torch.Size([2, 6, 3])

context_vec.shape:
torch.Size([2, 6, 2])


## Multi-Head-Attention

### `MultiHeadAttentionWrapper`

Wrapper implements n-heads with a list of `CausalAttention` (not efficient)

In [143]:
class MultiHeadAttentionWrapper(torch.nn.Module):
    def __init__(
        self,
        d_in,
        d_out,
        context_length,
        dropout,
        num_heads,
        qkv_bias=False
    ):
        super().__init__()
        self.heads = torch.nn.ModuleList(
            [CausalAttention(
                d_in, d_out, context_length, dropout, qkv_bias
            ) for _ in range(num_heads)]
        )

    def forward(self, x):
        # sequential processing
        return torch.cat([head(x) for head in self.heads], dim=-1)

#### Example: Using `MultiHeadAttentionWrapper`

`num_heads=2` times `d_out=2` results in a 4-dim context vector

In [144]:
torch.manual_seed(123)

# num of tokens
context_length = batch.shape[1]
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(
    d_in, d_out, context_length, 0.0, num_heads=2
)
context_vecs = mha(batch)

print(f"Context vectors:\n{context_vecs}")
print(f"\nShape of context vectors:\n{context_vecs.shape}")

Context vectors:
tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)

Shape of context vectors:
torch.Size([2, 6, 4])


### Efficient MultiHeadAttention

Raschka (2025), pp.116-117. 
Code take from: https://github.com/rasbt/LLMs-from-scratch/blob/main/ch03/01_main-chapter-code/ch03.ipynb

Single-Head-Attention-Layers aren't stacked above each other like before. Instead one Multi-Head-Attention-Layer is internally split into n-Attention-Heads

In [150]:
class MultiHeadAttention(torch.nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = torch.nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = torch.nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        # As in `CausalAttention`, for inputs where `num_tokens` exceeds `context_length`, 
        # this will result in errors in the mask creation further below. 
        # In practice, this is not a problem since the LLM (chapters 4-7) ensures that inputs  
        # do not exceed `context_length` before reaching this forward method.

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec


In [154]:
# (b, num_heads, num_tokens, head_dim) = (1, 2, 3, 4)
a = torch.tensor([[[[0.2745, 0.6584, 0.2775, 0.8573],
                    [0.8993, 0.0390, 0.9268, 0.7388],
                    [0.7179, 0.7058, 0.9156, 0.4340]],

                   [[0.0772, 0.3565, 0.1479, 0.5331],
                    [0.4066, 0.2318, 0.4545, 0.9737],
                    [0.4606, 0.5159, 0.4220, 0.5786]]]])

first_head = a[0,0,:,:]
first_res = first_head @ first_head.T
print(f"First head:\n{first_res}")

second_head = a[0,1,:,:]
second_res = second_head @ second_head.T
print(f"\nSecond head:\n{second_res}")

First head:
tensor([[1.3208, 1.1631, 1.2879],
        [1.1631, 2.2150, 1.8424],
        [1.2879, 1.8424, 2.0402]])

Second head:
tensor([[0.4391, 0.7003, 0.5903],
        [0.7003, 1.3737, 1.0620],
        [0.5903, 1.0620, 0.9912]])


In [152]:
torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])
